# PrimeKG AML Subgraph Extraction
## Goal: Extract a 2-hop subgraph around AML disease nodes from PrimeKG

This notebook will guide you through extracting an AML-related subgraph from the large PrimeKG dataset using memory-efficient techniques.

**Steps:**
1. First, we'll find AML disease nodes
2. Extract 1-hop neighbors (directly connected nodes)
3. Extract 2-hop neighbors (nodes connected to 1-hop neighbors)
4. Save the filtered subgraph

## Step 1: Import Libraries and Setup

In [2]:
import pandas as pd
import numpy as np
from collections import defaultdict
import gc  # For garbage collection to manage memory

print("Libraries imported successfully!")

Libraries imported successfully!


## Step 2: Find AML Disease Nodes

First, let's search for AML disease nodes in the knowledge graph. We'll use chunked reading to avoid loading the entire file into memory.

In [3]:
# Function to search for AML disease nodes using chunked reading
def find_disease_nodes(filename, disease_keywords, chunk_size=100000):
    """
    Search for disease nodes containing specific keywords
    Uses chunked reading to save memory
    """
    disease_nodes = set()
    
    print(f"Searching for disease nodes containing: {disease_keywords}")
    
    # Read file in chunks
    chunk_iter = pd.read_csv(filename, chunksize=chunk_size, low_memory=False)
    
    for i, chunk in enumerate(chunk_iter):
        # Check x_type for disease and x_name for keywords
        mask_x = (chunk['x_type'] == 'disease')
        for keyword in disease_keywords:
            mask_x = mask_x & chunk['x_name'].str.contains(keyword, case=False, na=False)
        
        # Check y_type for disease and y_name for keywords
        mask_y = (chunk['y_type'] == 'disease')
        for keyword in disease_keywords:
            mask_y = mask_y & chunk['y_name'].str.contains(keyword, case=False, na=False)
        
        # Collect matching disease IDs and names
        if mask_x.any():
            for _, row in chunk[mask_x].iterrows():
                disease_nodes.add((row['x_id'], row['x_name']))
        
        if mask_y.any():
            for _, row in chunk[mask_y].iterrows():
                disease_nodes.add((row['y_id'], row['y_name']))
        
        if (i + 1) % 10 == 0:
            print(f"Processed {(i+1)*chunk_size:,} rows...")
    
    return disease_nodes

# Search for AML (Acute Myeloid Leukemia) nodes
aml_keywords = ['acute', 'myeloid', 'leukemia']
aml_nodes = find_disease_nodes('kg.csv', aml_keywords, chunk_size=100000)

print(f"\nFound {len(aml_nodes)} AML-related disease nodes:")
for node_id, node_name in sorted(aml_nodes):
    print(f"  ID: {node_id}, Name: {node_name}")

Searching for disease nodes containing: ['acute', 'myeloid', 'leukemia']
Processed 1,000,000 rows...
Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Processed 5,000,000 rows...
Processed 6,000,000 rows...
Processed 6,000,000 rows...
Processed 7,000,000 rows...
Processed 7,000,000 rows...
Processed 8,000,000 rows...
Processed 8,000,000 rows...

Found 23 AML-related disease nodes:
  ID: 11118, Name: bilineal acute myeloid leukemia
  ID: 15164, Name: acute myeloid leukemia and myelodysplastic syndromes related to alkylating agent
  ID: 15165, Name: acute myeloid leukemia and myelodysplastic syndromes related to topoisomerase type 2 inhibitor
  ID: 15166, Name: acute myeloid leukemia with t(8;21)(q22;q22) translocation
  ID: 15608, Name: acute myeloid leukemia and myelodysplastic syndromes related to radiation
  ID: 

## Step 3: Extract 1-Hop Neighbors

Now we'll extract all edges directly connected to AML nodes (1-hop neighbors).

In [4]:
# Extract AML disease IDs
aml_ids = set([node_id for node_id, _ in aml_nodes])
print(f"AML Disease IDs: {aml_ids}")

# Function to extract 1-hop neighbors
def extract_1hop_edges(filename, seed_ids, chunk_size=100000):
    """
    Extract all edges connected to seed nodes (1-hop)
    Returns: edges_df, neighbor_ids
    """
    edges_list = []
    neighbor_ids = set()
    
    print(f"Extracting 1-hop neighbors for {len(seed_ids)} seed nodes...")
    
    chunk_iter = pd.read_csv(filename, chunksize=chunk_size, low_memory=False)
    
    for i, chunk in enumerate(chunk_iter):
        # Find edges where either x_id or y_id is in seed_ids
        mask = chunk['x_id'].isin(seed_ids) | chunk['y_id'].isin(seed_ids)
        
        if mask.any():
            matching_edges = chunk[mask]
            edges_list.append(matching_edges)
            
            # Collect neighbor IDs (nodes connected to seed nodes)
            neighbor_ids.update(matching_edges['x_id'].unique())
            neighbor_ids.update(matching_edges['y_id'].unique())
        
        if (i + 1) % 10 == 0:
            print(f"Processed {(i+1)*chunk_size:,} rows, found {len(edges_list)} chunks with edges...")
    
    # Combine all edge chunks
    if edges_list:
        edges_df = pd.concat(edges_list, ignore_index=True)
    else:
        edges_df = pd.DataFrame()
    
    print(f"Found {len(edges_df)} 1-hop edges")
    print(f"Found {len(neighbor_ids)} unique 1-hop neighbors")
    
    return edges_df, neighbor_ids

# Extract 1-hop edges and neighbors
hop1_edges, hop1_neighbors = extract_1hop_edges('kg.csv', aml_ids, chunk_size=100000)

print(f"\n1-Hop Summary:")
print(f"  Edges: {len(hop1_edges):,}")
print(f"  Unique nodes: {len(hop1_neighbors):,}")

# Free up memory
gc.collect()

AML Disease IDs: {'44923', '15166', '35112', '15165', '18435', '11118', '18433', '15164', '20078', '18434', '17893', '4996', '19457', '20316', '18436', '17894', '15667', '5223', '15608', '19456', '18437', '18256', '20317'}
Extracting 1-hop neighbors for 23 seed nodes...
Processed 1,000,000 rows, found 1 chunks with edges...
Processed 1,000,000 rows, found 1 chunks with edges...
Processed 2,000,000 rows, found 1 chunks with edges...
Processed 2,000,000 rows, found 1 chunks with edges...
Processed 3,000,000 rows, found 1 chunks with edges...
Processed 3,000,000 rows, found 1 chunks with edges...
Processed 4,000,000 rows, found 7 chunks with edges...
Processed 4,000,000 rows, found 7 chunks with edges...
Processed 5,000,000 rows, found 7 chunks with edges...
Processed 5,000,000 rows, found 7 chunks with edges...
Processed 6,000,000 rows, found 9 chunks with edges...
Processed 6,000,000 rows, found 9 chunks with edges...
Processed 7,000,000 rows, found 13 chunks with edges...
Processed 7,0

105

## Step 4: Extract 2-Hop Neighbors

Now we'll extract edges connected to the 1-hop neighbors to get the 2-hop subgraph.

In [5]:
# Function to extract 2-hop edges (edges connected to 1-hop neighbors)
def extract_2hop_edges(filename, hop1_neighbors, chunk_size=100000):
    """
    Extract all edges connected to 1-hop neighbors (2-hop from seed)
    """
    edges_list = []
    hop2_neighbors = set()
    
    print(f"Extracting 2-hop neighbors from {len(hop1_neighbors)} 1-hop nodes...")
    print("This may take a few minutes...")
    
    chunk_iter = pd.read_csv(filename, chunksize=chunk_size, low_memory=False)
    
    for i, chunk in enumerate(chunk_iter):
        # Find edges where either x_id or y_id is in hop1_neighbors
        mask = chunk['x_id'].isin(hop1_neighbors) | chunk['y_id'].isin(hop1_neighbors)
        
        if mask.any():
            matching_edges = chunk[mask]
            edges_list.append(matching_edges)
            
            # Collect 2-hop neighbor IDs
            hop2_neighbors.update(matching_edges['x_id'].unique())
            hop2_neighbors.update(matching_edges['y_id'].unique())
        
        if (i + 1) % 10 == 0:
            print(f"Processed {(i+1)*chunk_size:,} rows...")
    
    # Combine all edge chunks
    if edges_list:
        edges_df = pd.concat(edges_list, ignore_index=True)
    else:
        edges_df = pd.DataFrame()
    
    print(f"Found {len(edges_df)} 2-hop edges")
    print(f"Found {len(hop2_neighbors)} unique nodes in 2-hop neighborhood")
    
    return edges_df, hop2_neighbors

# Extract 2-hop edges
hop2_edges, hop2_neighbors = extract_2hop_edges('kg.csv', hop1_neighbors, chunk_size=100000)

print(f"\n2-Hop Summary:")
print(f"  Total edges: {len(hop2_edges):,}")
print(f"  Unique nodes: {len(hop2_neighbors):,}")

# Free up memory
gc.collect()

Extracting 2-hop neighbors from 336 1-hop nodes...
This may take a few minutes...
Processed 1,000,000 rows...
Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Processed 5,000,000 rows...
Processed 6,000,000 rows...
Processed 6,000,000 rows...
Processed 7,000,000 rows...
Processed 7,000,000 rows...
Processed 8,000,000 rows...
Processed 8,000,000 rows...
Found 141779 2-hop edges
Found 35082 unique nodes in 2-hop neighborhood

2-Hop Summary:
  Total edges: 141,779
  Unique nodes: 35,082
Found 141779 2-hop edges
Found 35082 unique nodes in 2-hop neighborhood

2-Hop Summary:
  Total edges: 141,779
  Unique nodes: 35,082


0

## Step 5: Save the AML Subgraph

Save the extracted 2-hop subgraph to a new CSV file.

In [6]:
# Remove duplicate edges (keep unique edges only)
aml_subgraph = hop2_edges.drop_duplicates()

print(f"AML 2-hop Subgraph Statistics:")
print(f"  Total edges: {len(aml_subgraph):,}")
print(f"  Unique nodes: {len(hop2_neighbors):,}")
print(f"  Memory usage: {aml_subgraph.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Save to CSV
output_dir = '../../interim/primekg'
import os
os.makedirs(output_dir, exist_ok=True)
output_file = f"{output_dir}/aml_2hop_subgraph.csv"
aml_subgraph.to_csv(output_file, index=False)
print(f"\n✓ Subgraph saved to: {output_file}")

# Check file size
file_size_mb = os.path.getsize(output_file) / 1024**2
print(f"  File size: {file_size_mb:.2f} MB")

AML 2-hop Subgraph Statistics:
  Total edges: 141,779
  Unique nodes: 35,082
  Memory usage: 91.72 MB
  Memory usage: 91.72 MB

✓ Subgraph saved to: aml_2hop_subgraph.csv
  File size: 16.66 MB

✓ Subgraph saved to: aml_2hop_subgraph.csv
  File size: 16.66 MB


## Step 6: Analyze the Subgraph

Let's examine what types of relationships and nodes are in our AML subgraph.

In [7]:
# Analyze relation types
print("Relation Types in AML Subgraph:")
print(aml_subgraph['relation'].value_counts())

print("\n" + "="*60)

# Analyze node types
x_types = aml_subgraph['x_type'].value_counts()
y_types = aml_subgraph['y_type'].value_counts()
all_types = pd.concat([x_types, y_types]).groupby(level=0).sum().sort_values(ascending=False)

print("\nNode Types in AML Subgraph:")
print(all_types)

print("\n" + "="*60)

# Show some sample edges
print("\nSample edges from the subgraph:")
print(aml_subgraph[['relation', 'x_name', 'x_type', 'y_name', 'y_type']].head(10))

Relation Types in AML Subgraph:
relation
drug_drug                     44224
disease_protein               38850
molfunc_protein               26251
bioprocess_protein             7297
drug_effect                    3654
protein_protein                3158
anatomy_protein_present        3016
pathway_protein                2683
disease_phenotype_positive     2553
disease_disease                1878
drug_protein                   1624
indication                     1396
contraindication               1252
bioprocess_bioprocess           852
phenotype_phenotype             748
anatomy_anatomy                 695
off-label use                   518
phenotype_protein               346
exposure_disease                306
molfunc_molfunc                 170
cellcomp_protein                 96
exposure_protein                 76
cellcomp_cellcomp                58
disease_phenotype_negative       40
exposure_bioprocess              20
pathway_pathway                  12
anatomy_protein_absent 

## Optional: Create Node List

Extract unique nodes with their properties for further analysis.

In [8]:
# Create a node list with unique nodes
x_nodes = aml_subgraph[['x_id', 'x_name', 'x_type', 'x_source']].rename(
    columns={'x_id': 'node_id', 'x_name': 'node_name', 
             'x_type': 'node_type', 'x_source': 'node_source'})

y_nodes = aml_subgraph[['y_id', 'y_name', 'y_type', 'y_source']].rename(
    columns={'y_id': 'node_id', 'y_name': 'node_name', 
             'y_type': 'node_type', 'y_source': 'node_source'})

# Combine and remove duplicates
nodes = pd.concat([x_nodes, y_nodes]).drop_duplicates().reset_index(drop=True)

print(f"Total unique nodes: {len(nodes):,}")
print(f"\nNode types distribution:")
print(nodes['node_type'].value_counts())

# Save nodes list
nodes_file = f"{output_dir}/aml_2hop_nodes.csv"
nodes.to_csv(nodes_file, index=False)
print(f"\n✓ Nodes list saved to: {nodes_file}")

# Show sample nodes
print("\nSample nodes:")
print(nodes.head(10))

Total unique nodes: 37,671

Node types distribution:
node_type
gene/protein          26192
drug                   3027
disease                2806
biological_process     2194
effect/phenotype       1406
anatomy                1070
pathway                 646
molecular_function      141
exposure                128
cellular_component       61
Name: count, dtype: int64

✓ Nodes list saved to: aml_2hop_nodes.csv

Sample nodes:
  node_id node_name     node_type node_source
0    4088     SMAD3  gene/protein        NCBI
1    5460    POU5F1  gene/protein        NCBI
2    5906     RAP1A  gene/protein        NCBI
3    5515    PPP2CA  gene/protein        NCBI
4   26063     DECR2  gene/protein        NCBI
5    1627      DBN1  gene/protein        NCBI
6      18      ABAT  gene/protein        NCBI
7     648      BMI1  gene/protein        NCBI
8    5292      PIM1  gene/protein        NCBI
9    6711    SPTBN1  gene/protein        NCBI


## Summary

You've successfully extracted a 2-hop AML subgraph! 

**Files created:**
- `aml_2hop_subgraph.csv` - Edge list of the subgraph
- `aml_2hop_nodes.csv` - List of unique nodes

**What this means:**
- **Hop 0**: AML disease nodes (seed nodes)
- **Hop 1**: All nodes directly connected to AML (genes, drugs, pathways, etc.)
- **Hop 2**: All nodes connected to the 1-hop neighbors

This subgraph contains all relationships within 2 steps from AML, which is much smaller and manageable for your 16GB RAM laptop!